# Manual Testing: QueryOrchestrator → TavilyRetriever Pipeline

This notebook provides hands-on testing of the integrated QueryOrchestrator and TavilyRetriever agents using real API keys.

## Pipeline Flow
1. **QueryOrchestrator** - Parses user query → structured search query + intent
2. **TavilyRetriever** - Executes Tavily two-step process (search → extract)
3. **State Management** - Track agent execution and results

## Requirements
- Real OPENAI_API_KEY and TAVILY_API_KEY in .env file
- Backend dependencies installed

## Setup and Imports

In [1]:
import os
import sys
import asyncio
import json
from datetime import datetime
from pprint import pprint

# Add backend to path
sys.path.append('..')

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

print("✅ Environment loaded")
print(f"OPENAI_API_KEY: {'✅ Set' if os.getenv('OPENAI_API_KEY') else '❌ Missing'}")
print(f"TAVILY_API_KEY: {'✅ Set' if os.getenv('TAVILY_API_KEY') else '❌ Missing'}")

✅ Environment loaded
OPENAI_API_KEY: ✅ Set
TAVILY_API_KEY: ✅ Set


In [2]:
# Import our agents and state management
from app.agents.query_orchestrator_agent import QueryOrchestratorAgent
from app.agents.tavily_retriever_agent import TavilyRetrieverAgent
from app.agents.state import create_initial_state, get_state_summary
from app.config import settings

print("✅ Agents imported successfully")
print(f"Environment: {settings.ENVIRONMENT}")
print(f"OpenAI Model: {settings.OPENAI_MODEL}")

✅ Agents imported successfully
Environment: development
OpenAI Model: gpt-4o-mini


## Initialize Agents

In [3]:
# Create agent instances
query_agent = QueryOrchestratorAgent()
tavily_agent = TavilyRetrieverAgent()

print("✅ Agents initialized:")
print(f"- {query_agent.name} (OpenAI: {query_agent.llm.model_name})")
print(f"- {tavily_agent.name} (Client: {type(tavily_agent.tavily_client).__name__})")

[Query Orchestrator] Initialized Query Orchestrator with llm_provider: OpenAI and model: gpt-4o-mini
[Tavily Retriever] Initialized with development configuration
✅ Agents initialized:
- Query Orchestrator (OpenAI: gpt-4o-mini)
- Tavily Retriever (Client: OptimizedTavilyClient)


## Test Configuration

In [4]:
# Test Tavily configuration
config_test = await tavily_agent.test_configuration()
print("🔧 Tavily Configuration:")
pprint(config_test)

[Tavily Retriever] Testing Tavily configuration
[Tavily Retriever] Configuration test successful
🔧 Tavily Configuration:
{'agent_name': 'Tavily Retriever',
 'api_key_available': True,
 'client_type': 'OptimizedTavilyClient',
 'config': {'coverage_threshold': 0.6,
            'enable_fallback': False,
            'enable_intent_optimization': True,
            'enable_map_api': False,
            'enable_quality_filter': False,
            'map_max_depth': 1,
            'map_max_results': 10,
            'max_concurrent': 2,
            'max_results': 5,
            'min_domain_quality': 0.5,
            'query_max_length': 400,
            'search_depth': 'basic'}}


## Manual Test Functions

In [5]:
async def test_query_pipeline(raw_query: str, run_id: str = None):
    """
    Complete pipeline test: QueryOrchestrator → TavilyRetriever
    """
    if not run_id:
        run_id = f"manual_test_{datetime.now().strftime('%H%M%S')}"
    
    print(f"\n🚀 Testing Query: '{raw_query}'")
    print("=" * 60)
    
    # Create initial state
    state = create_initial_state(raw_query=raw_query, run_id=run_id)
    print(f"📋 Initial state created (run_id: {run_id})")
    
    try:
        # Step 1: QueryOrchestrator
        print("\n1️⃣ QueryOrchestrator Processing...")
        start_time = datetime.now()
        state = await query_agent.process(state)
        query_time = (datetime.now() - start_time).total_seconds()
        
        search_query = state.get("search_query")
        if search_query:
            print(f"✅ Query parsed successfully ({query_time:.2f}s)")
            print(f"   Intent: {search_query.intent}")
            print(f"   Category: {search_query.category}")
            print(f"   Normalized: '{search_query.normalized_query}'")
            if search_query.budget_max:
                print(f"   Budget: ${search_query.budget_max}")
            if search_query.constraints:
                print(f"   Constraints: {search_query.constraints}")
                
            # Show Tavily parameters
            tavily_params = state.get("tavily_search_params", {})
            print(f"   Tavily params: {tavily_params}")
        else:
            print("❌ Query parsing failed")
            return state
        
        # Step 2: TavilyRetriever
        print("\n2️⃣ TavilyRetriever Processing...")
        start_time = datetime.now()
        state = await tavily_agent.process(state)
        tavily_time = (datetime.now() - start_time).total_seconds()
        
        # Analyze results
        search_results = state.get("raw_search_results", [])
        extracted_content = state.get("extracted_content", [])
        coverage_score = state.get("coverage_score", 0.0)
        
        print(f"✅ Tavily processing completed ({tavily_time:.2f}s)")
        print(f"   Search results: {len(search_results)}")
        print(f"   Extracted content: {len(extracted_content)}")
        print(f"   Coverage score: {coverage_score:.2f}")
        
        # Show agent execution summary
        summary = get_state_summary(state)
        print(f"\n📊 Execution Summary:")
        print(f"   Total time: {query_time + tavily_time:.2f}s")
        print(f"   Agents completed: {summary['progress']['agents_completed']}")
        print(f"   Total cost: ${summary['progress']['total_cost_usd']:.4f}")
        
        return state
        
    except Exception as e:
        print(f"❌ Pipeline error: {e}")
        import traceback
        traceback.print_exc()
        return state

def show_search_results(state: dict, max_results: int = 3):
    """
    Display search results in a readable format
    """
    search_results = state.get("raw_search_results", [])
    
    if not search_results:
        print("No search results found")
        return
    
    # Ensure search_results is a list
    if isinstance(search_results, dict):
        search_results = search_results.get("results", [])
    
    if not isinstance(search_results, list):
        print(f"Error: search_results is not a list, got {type(search_results)}")
        return
    
    print(f"\n🔍 Search Results (showing {min(max_results, len(search_results))} of {len(search_results)}):")
    print("=" * 60)
    
    for i, result in enumerate(search_results[:max_results]):
        print(f"\n{i+1}. {result.get('title', 'No title')}")
        print(f"   URL: {result.get('url', 'No URL')}")
        content = result.get('content', 'No content')
        if len(content) > 150:
            content = content[:150] + "..."
        print(f"   Content: {content}")
        if 'score' in result:
            print(f"   Score: {result['score']:.3f}")

def show_extracted_content(state: dict, max_content: int = 2):
    """
    Display extracted content in a readable format
    """
    extracted_content = state.get("extracted_content", [])
    
    if not extracted_content:
        print("No extracted content found")
        return
    
    # Ensure extracted_content is a list
    if not isinstance(extracted_content, list):
        print(f"Error: extracted_content is not a list, got {type(extracted_content)}")
        return
    
    print(f"\n📄 Extracted Content (showing {min(max_content, len(extracted_content))} of {len(extracted_content)}):")
    print("=" * 60)
    
    for i, content in enumerate(extracted_content[:max_content]):
        print(f"\n{i+1}. URL: {content.get('url', 'No URL')}")
        extracted_text = content.get('content', 'No content')
        if len(extracted_text) > 300:
            extracted_text = extracted_text[:300] + "..."
        print(f"   Content: {extracted_text}")
        if 'quality_score' in content:
            print(f"   Quality: {content['quality_score']:.3f}")

## Interactive Testing

Now you can test different queries manually. Try the examples below or create your own!

### Test 1: Gaming Laptop Search

In [6]:
# Test gaming laptop search
result_state = await test_query_pipeline("best gaming laptop under $2000")


🚀 Testing Query: 'best gaming laptop under $2000'
📋 Initial state created (run_id: manual_test_225141)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'best gaming laptop under $2000'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "best gaming laptop under $2000",
  "normalized_query": "best gaming laptop",
  "intent": "product_search",
  "category": "laptop",
  "brand": null,
  "budget_min": null,
  "budget_max": 2000.0,
  "constraints": [
    "gaming"
  ],
  "priorities": [
    "performance"
  ],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "best gaming laptop",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (6.47s)
   Intent: product_search
   Category: laptop
   Normalized: 'best gaming laptop'
   Budget: $2000.0
   Constraints: ['gaming']
   Tavily params: {'query': 'best gaming laptop', 'search_depth': 'advanced', 'max_re

In [7]:
# Show detailed results
show_search_results(result_state, max_results=5)
show_extracted_content(result_state, max_content=3)


🔍 Search Results (showing 5 of 5):

1. Micro Center Platinum Collection – Top-Tier Gaming For ...
   URL: https://community.microcenter.com/discussion/9405/micro-center-platinum-collection-top-of-the-line-gaming-for-every-budget
   Content: The $1,000 mark is the sweet spot for gaming laptops. They come with that bit more performance, a bit more display clarity and brightness, all wrapped...
   Score: 0.715

2. Intel 10th Generation Core i7 and AMD Ryzen 7 Gaming ...
   URL: https://www.bestbuy.com/site/searchpage.jsp?_dyncharset=UTF-8&browsedCategory=pcmcat287600050003&id=pcat17071&iht=n&ks=960&list=y&qp=child_processormodelsv_facet%3Dname~Intel%2010th%20Generation%20Core%20i7%5Eparent_processormodelsv_facet%3Dname~AMD%20Ryzen%207&sc=Global&st=categoryid%24pcmcat287600050003&type=page&usc=All%20Categories
   Content: Image 19: PC Game Pass Doom The Dark Ages.      ASUS - TUF Gaming A16 16" FHD+ 165Hz Gaming Laptop - AMD Ryzen 9 - 32GB RAM - NVIDIA GeForce RTX 5070 ...
   Score: 0.536

### Test 2: Product Comparison

In [8]:
# Test product comparison
result_state = await test_query_pipeline("iPhone 15 vs Samsung Galaxy S24 camera quality")


🚀 Testing Query: 'iPhone 15 vs Samsung Galaxy S24 camera quality'
📋 Initial state created (run_id: manual_test_225306)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'iPhone 15 vs Samsung Galaxy S24 camera quality'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "iPhone 15 vs Samsung Galaxy S24 camera quality",
  "normalized_query": "iPhone 15 Samsung Galaxy S24 camera quality",
  "intent": "comparison",
  "category": "smartphone",
  "brand": null,
  "budget_min": null,
  "budget_max": null,
  "constraints": [
    "camera quality"
  ],
  "priorities": [
    "camera"
  ],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "iPhone 15 Samsung Galaxy S24 camera quality",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (1.79s)
   Intent: comparison
   Category: smartphone
   Normalized: 'iPhone 15 Samsung Galaxy S24 camera quality'
   Cons

In [9]:
# Show detailed results
show_search_results(result_state, max_results=4)
show_extracted_content(result_state, max_content=2)


🔍 Search Results (showing 4 of 5):

1. Samsung Galaxy S24 Ultra
   URL: https://www.notebookcheck.net/Samsung-Galaxy-S24-Ultra.808695.0.html
   Content: The Samsung Galaxy S24 Ultra delivered a decent performance in the DXOMARK Camera tests and was a slight overall improvement over its predecessor S23 ...
   Score: 0.794

2. Compare - Samsung Galaxy S24
   URL: https://www.gsmarena.com/compare.php3?idPhone1=12773&idPhone2=13964&idPhone3=13610
   Content: | Main Camera | Modules | 50 MP, f/1.8, 24mm (wide), 1/1.56", 1.0µm, dual pixel PDAF, OIS 10 MP, f/2.4, 67mm (telephoto), 1/3.94", 1.0µm, PDAF, OIS, 3...
   Score: 0.723

3. Samsung Galaxy S24 FE - Full phone specifications
   URL: https://www.gsmarena.com/samsung_galaxy_s24_fe-13262.php
   Content: | Main Camera | Triple | 50 MP, f/1.8, 24mm (wide), 1/1.57", 1.0µm, dual pixel PDAF, OIS 8 MP, f/2.4, 75mm (telephoto), 1/4.4", 1.0µm, PDAF, OIS, 3x o...
   Score: 0.436

4. Apple iPhone 17 Pro Max vs Samsung Galaxy S25 Ultra
   URL: https

### Test 3: Review Search

In [10]:
# Test review search
result_state = await test_query_pipeline("Sony WH-1000XM5 headphones review")


🚀 Testing Query: 'Sony WH-1000XM5 headphones review'
📋 Initial state created (run_id: manual_test_225412)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'Sony WH-1000XM5 headphones review'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "Sony WH-1000XM5 headphones review",
  "normalized_query": "Sony WH-1000XM5 headphones",
  "intent": "review_search",
  "category": "headphones",
  "brand": "Sony",
  "budget_min": null,
  "budget_max": null,
  "constraints": [],
  "priorities": [],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "Sony WH-1000XM5 headphones",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (1.92s)
   Intent: review_search
   Category: headphones
   Normalized: 'Sony WH-1000XM5 headphones'
   Tavily params: {'query': 'Sony WH-1000XM5 headphones', 'search_depth': 'advanced', 'max_results': 10}

2️⃣ TavilyRetriever Pro

In [11]:
# Show detailed results
show_search_results(result_state, max_results=3)
show_extracted_content(result_state, max_content=2)


🔍 Search Results (showing 3 of 5):

1. The Sony WH-1000XM5 just hit a record low price at ...
   URL: https://www.tomsguide.com/audio/headphones/the-sony-wh-1000xm5-just-hit-a-record-price-low-at-amazon-this-is-incredible
   Content: The Sony WH-1000XM5 are our choice for the best headphones you can buy. They earned an almost perfect 4.5-star rating in our Sony WH-1000XM5 review, w...
   Score: 0.827

2. Should you buy the Sony WH-1000XM5 headphones in ...
   URL: https://www.techradar.com/audio/headphones/should-you-buy-the-sony-wh-1000xm5-headphones-in-2025
   Content: Since their release in 2022, the Sony WH-1000XM5 over-ear headphones have ranked as elite contenders for our lists of the best noise-cancelling headph...
   Score: 0.758

3. Best Noise-Canceling Headphones of 2025: AirPods, Bose ...
   URL: https://www.cnet.com/tech/mobile/best-noise-canceling-headphones/
   Content: Upgraded design with more comfortable fit
 Improved noise canceling and sound quality
 New QN3 chip is

### Custom Query Testing

Use this cell to test your own queries:

In [12]:
# Your custom query here
custom_query = "mechanical keyboard for programming Cherry MX switches"
custom_result = await test_query_pipeline(custom_query)


🚀 Testing Query: 'mechanical keyboard for programming Cherry MX switches'
📋 Initial state created (run_id: manual_test_225507)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'mechanical keyboard for programming Cherry MX switches'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "mechanical keyboard for programming Cherry MX switches",
  "normalized_query": "mechanical keyboard Cherry MX switches programming",
  "intent": "product_search",
  "category": "keyboard",
  "brand": null,
  "budget_min": null,
  "budget_max": null,
  "constraints": [
    "Cherry MX switches",
    "for programming"
  ],
  "priorities": [
    "performance"
  ],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "mechanical keyboard Cherry MX switches programming",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (2.14s)
   Intent: product_search
   Category: key

In [13]:
# Show your custom results
show_search_results(custom_result, max_results=3)
show_extracted_content(custom_result, max_content=2)


🔍 Search Results (showing 3 of 5):

1. CORSAIR K100 RGB Full size Wired Mechanical Cherry ...
   URL: https://www.bestbuy.com/product/corsair-k100-rgb-full-size-wired-mechanical-cherry-mx-speed-switch-gaming-keyboard-with-elgato-stream-deck-software-integration-black/J39QHHFZCP
   Content: CHERRY MX SPEED RGB Silver mechanical keyswitches, guaranteed for 100 million keypresses, offer ultra-fast 1.2mm actuation while registering keypresse...
   Score: 0.489

2. Mechanical Gaming Keyboards
   URL: https://www.bestbuy.com/site/searchpage.jsp?browsedCategory=pcmcat1661795797857&id=pcat17071&qp=brand_facet%3DBrand%7ECORSAIR&st=pcmcat1661795797857_categoryid%24pcmcat304600050014
   Content: CORSAIR - K70 RGB PRO Full-size Wired Mechanical Cherry MX Speed Linear Switch Gaming Keyboard with PBT Double-Shot Keycaps - Black.
   Score: 0.474

3. IKBC / PC Gaming Keyboards / PC Accessories
   URL: https://www.amazon.com/PC-Gaming-Keyboards-IKBC-Accessories/s?keywords=PC+Gaming+Keyboards&rh=n%3A40

## Deep Dive: State Analysis

Examine the complete state structure and agent execution details:

In [14]:
# Analyze the complete state from your last test
print("🔍 Complete State Analysis:")
print("=" * 60)

# Show state keys
print(f"State keys: {list(custom_result.keys())}")

# Show search query details
search_query = custom_result.get("search_query")
if search_query:
    print(f"\n📝 Parsed Query:")
    pprint(search_query.model_dump())

# Show agent execution steps
agent_steps = custom_result.get("agent_steps", [])
print(f"\n🏃 Agent Execution Steps ({len(agent_steps)}):")
for step in agent_steps:
    print(f"  {step.agent_name}: {step.status} ({step.execution_time_ms}ms, ${step.cost_usd:.4f})")

🔍 Complete State Analysis:
State keys: ['raw_query', 'user_id', 'run_id', 'search_query', 'candidate_urls', 'filtered_urls', 'source_plan', 'raw_listings', 'raw_reviews', 'canonical_products', 'price_aggregations', 'ranked_products', 'agent_steps', 'total_cost_usd', 'execution_time_ms', 'errors', 'warnings', 'websocket_manager', 'job_id', 'tavily_search_params', 'raw_search_results', 'extracted_content', 'coverage_score']

📝 Parsed Query:
{'brand': None,
 'budget_max': None,
 'budget_min': None,
 'category': 'keyboard',
 'constraints': ['Cherry MX switches', 'for programming'],
 'intent': 'product_search',
 'normalized_query': 'mechanical keyboard Cherry MX switches programming',
 'priorities': ['performance'],
 'raw_query': 'mechanical keyboard for programming Cherry MX switches',
 'region': 'US'}

🏃 Agent Execution Steps (2):
  Query Orchestrator: success (2140ms, $0.0000)
  Tavily Retriever: success (12845ms, $0.0000)


## Performance Testing

Test multiple queries to understand performance patterns:

In [15]:
# Performance test with multiple queries
test_queries = [
    "wireless earbuds under $100",
    "RTX 4070 graphics card review",
    "MacBook Air vs ThinkPad comparison",
    "best monitor for productivity 4K"
]

performance_results = []

for i, query in enumerate(test_queries, 1):
    print(f"\n{i}/{len(test_queries)}: Testing '{query}'")
    start_time = datetime.now()
    
    result = await test_query_pipeline(query, f"perf_test_{i}")
    
    total_time = (datetime.now() - start_time).total_seconds()
    summary = get_state_summary(result)
    
    performance_results.append({
        "query": query,
        "time_seconds": total_time,
        "cost_usd": summary['progress']['total_cost_usd'],
        "search_results": len(result.get("raw_search_results", [])),
        "extracted_content": len(result.get("extracted_content", []))
    })

print("\n📊 Performance Summary:")
print("=" * 60)
for result in performance_results:
    print(f"Query: {result['query'][:40]}...")
    print(f"  Time: {result['time_seconds']:.1f}s, Cost: ${result['cost_usd']:.4f}")
    print(f"  Results: {result['search_results']} search, {result['extracted_content']} extracted\n")


1/4: Testing 'wireless earbuds under $100'

🚀 Testing Query: 'wireless earbuds under $100'
📋 Initial state created (run_id: perf_test_1)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'wireless earbuds under $100'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "wireless earbuds under $100",
  "normalized_query": "wireless earbuds",
  "intent": "product_search",
  "category": "headphones",
  "brand": null,
  "budget_min": null,
  "budget_max": 100.0,
  "constraints": [],
  "priorities": [
    "price"
  ],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "wireless earbuds",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (1.95s)
   Intent: product_search
   Category: headphones
   Normalized: 'wireless earbuds'
   Budget: $100.0
   Tavily params: {'query': 'wireless earbuds', 'search_depth': 'advanced', 'max_results': 10}

2️⃣ Tavily

## Experimentation Area

Use this space to experiment with different aspects of the pipeline:

In [16]:
# Experiment with edge cases
edge_cases = [
    "laptop",  # Very simple query
    "best affordable gaming laptop with RTX 4060 under $1200 for programming and gaming",  # Complex query
    "xyz123 unknown product",  # Non-existent product
    ""  # Empty query
]

print("🧪 Testing Edge Cases:")
for query in edge_cases:
    if not query:
        query = "[empty string]"
    print(f"\nTesting: '{query}'")
    try:
        result = await test_query_pipeline(query if query != "[empty string]" else "")
        print("✅ Handled successfully")
    except Exception as e:
        print(f"❌ Error: {e}")

🧪 Testing Edge Cases:

Testing: 'laptop'

🚀 Testing Query: 'laptop'
📋 Initial state created (run_id: manual_test_225810)

1️⃣ QueryOrchestrator Processing...
[Query Orchestrator] Starting query parsing
[Query Orchestrator] Parsing query: 'laptop'
[Query Orchestrator] Successfully parsed query: {
  "raw_query": "laptop",
  "normalized_query": "laptop",
  "intent": "product_search",
  "category": "laptop",
  "brand": null,
  "budget_min": null,
  "budget_max": null,
  "constraints": [],
  "priorities": [],
  "region": "US"
}
[Query Orchestrator] Generated Tavily search params: {
  "query": "laptop",
  "search_depth": "advanced",
  "max_results": 10
}
✅ Query parsed successfully (1.57s)
   Intent: product_search
   Category: laptop
   Normalized: 'laptop'
   Tavily params: {'query': 'laptop', 'search_depth': 'advanced', 'max_results': 10}

2️⃣ TavilyRetriever Processing...
[Tavily Retriever] Starting Tavily search and extraction
[Tavily Retriever] Processing query: 'laptop' (intent: produ

Tavily search failed: Query is missing.


[Tavily Retriever] Error in Tavily processing: Query is missing.
✅ Tavily processing completed (0.39s)
   Search results: 0
   Extracted content: 0
   Coverage score: 0.00

📊 Execution Summary:
   Total time: 0.39s
   Agents completed: 1
   Total cost: $0.0000
✅ Handled successfully


## Next Steps

After testing this pipeline, the next agent to implement is:
- **CredibilityFilterAgent** - Filter and score results by domain credibility and recency

The complete pipeline will be:
`QueryOrchestrator → RetrievalSplitter → TavilyRetriever → CredibilityFilter → SpecExtractor → ResultsRanker`